<h1 style="font-size: 40px; margin-bottom: 0px;">10.2 RNA-seq counts</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

In notebook 10-1, we aligned our RNA-seq reads to a reference genome using the splice aware aligner HISAT2, and today, we'll be counting our mapped reads in order to set up for our subsequent differential expression analysis. To do this, we'll make use of a program called <code>featureCounts</code>, which will allow us to count the reads mapped to different genomic features, and in the case of RNA-seq, we're interested in counting reads that have mapped to exons. Each group will then obtain raw counts for their control and TAZ KO replicates. We'll take a look at the data in our Python notebooks to do some basic QC, and then take a look to see if we might have some potentially differentially expressed genes just solely based on one replicate. Like for 9-1 and 10-1, we'll set up our scripts, then we can breakdown the code and concepts while our script runs.

We'll then work on a Python script that we can incorporate into the analysis pipeline that we've beeen setting up so far, so our scripts will be able to proceed through all the RNA-seq analysis steps that we've gone through up until this point.

<strong>Learning objectives:</strong>

<ul>
    <li>Setting up checkpoints in bash scripts</li>
    <li>Incorporate read counting into our scripts</li>
    <li>Run a QC on read counts for your group's replicate</li>
</ul>

<h1 style="font-size: 40px; margin-bottom: 0px;">Counting reads with <code>featureCounts</code></h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

<a href="https://subread.sourceforge.net/featureCounts.html" rel="noopener noreferrer"><u>Documentation for <code>featureCounts</code> can be found here.</u></a>

<code>featureCounts</code> is developed by <a href="https://academic.oup.com/bioinformatics/article-lookup/doi/10.1093/bioinformatics/btt656" rel="noopener noreferrer"><u>Liao et al 2014.</u></a> <code>featureCounts</code> is a package for counting mapped reads based on their alignment with genomic features, such as genes or exons or other genomic features, and <code>featureCounts</code> is part of the larger <code>Subread</code> package for processing NGS datasets. Here, we'll be making use of <code>featureCounts</code> for quantifying our read counts from our RNA-seq data in order to identify differentially expressed genes. After you obtain your group's count data, you'll use this Python notebook to play around with the outputs to run some quick QC.

Another commonly used package for counting reads is called <code>HTSeq</code>, which is a Python package for working with high-throughput sequencing data in Python developed by Fabio Zanini, Simon Anders, Givanna Putri and contributors (<a href="https://academic.oup.com/bioinformatics/article/38/10/2943/6551247" rel="noopener noreferrer"><u>Anders et al 2014</u></a>, <a href="https://academic.oup.com/bioinformatics/article/31/2/166/2366196" rel="noopener noreferrer"><u>Putri et al 2022</u></a>).

For those of you working ahead, we'll be adding to our bash script that we set up as part of notebook 10-1.

<h2>1. Pull out some rows from our annotation file to look at later</h2>

Genomic features are usually annotated in a GFF or GTF format, which is a fairly large tab-delimited table containing the following information for each annotated genomic feature:

<table style="text-align: center;">
    <tr style="font-size: 15px;">
        <th>seqname</th>
        <th>source</th>
        <th>feature</th>
        <th>start</th>
        <th>end</th>
        <th>score</th>
        <th>strand</th>
        <th>frame</th>
        <th>attribute</th>
    </tr>
    <tr>
        <td>Chr name</td>
        <td>Program that generated the data or the data source</td>
        <td>type of feature (gene, exon, cds, etc)</td>
        <td>start position</td>
        <td>end position</td>
        <td>floating point value</td>
        <td>+ or -</td>
        <td>codon frame</td>
        <td>additional information for each feature</td>
    </tr>
</table>

For this step, we'll incorporate into our shell script a quick operation to pull out some rows from our GTF file so that we can take a look at it later to better understand the information it contains.

In [ ]:
%%bash


<h2>2. Make a <code>counts</code> directory to hold outputs</h2>

To keep things organized, we'll make a directory to hold the outputs from our read counting.

In [ ]:
%%bash


<h2>3. Set up <code>featureCounts</code> to obtain read counts</h2>

Here, we will make use of <code>featureCounts</code> to count the reads that overlap with exons in order to estimate the levels of gene expression in our control and TAZ KO cells.

We will set up in our script the command line input to take a gene annotation file and our alignment files to then quantify how many reads map to each annotated gene.

<a href="https://subread.sourceforge.net/SubreadUsersGuide.pdf" rel="noopener noreferrer"><u>Detailed documentation for <code>featureCounts</code> can be found here with Table 3 indicating arguments used by <code>featureCounts</code>.</u></a>

In [ ]:
%%bash


Now let's run our script, and we can review the associated concepts and breakdown the code.

<h2>Let's break down the code:</h2>

```bash
featureCounts
```

This is to call up <code>featureCounts</code> to count reads mapped to a set of genomic features.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
-t exon
```

The <code>-t</code> option allows us to specify what type of feature from the <strong>feature</strong> column of the GTF file that we want to obtain read counts for. In this setup, we are looking to determine how many reads overlap with annotated exons.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
-g gene_id
```

The <code>-g</code> option provides the labels for each exon that we determine the counts for. Here, we specify that we want it to pull the <strong>gene_id</strong>, and use it as the label for its respective exon.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
-p
```

Here, we specify that our BAM file contains reads that come from alignment of paired-end reads.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
--countReadPairs
```

We then specify that we want to quantify based on read pairs instead of individual reads, essentially specifying that we want to count sequenced fragments instead of each sequence read.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
-s 2
```

This option tells <code>featureCounts</code> the strandedness of your cDNA library, and it's an important option because it will have a noticeable impact on the raw counts that you get. If you select the incorrect strandedness, you will end up with inaccurate quantifications that will impact your differential expression analysis. 

Since our cDNA library is directional on the first strand, so we will need to specify <code>2</code> for the strandedness option.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
-T 4
```

We can specify a performance option like we've done before for <code>bowtie2</code>, <code>samtools</code>, and <code>hisat2</code>. Here, we specify that we want to use four parallel threads to perform our read count.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
-a ~/_shared/bcourses-1557888-readonly/rna-seq/rna-feature/hg19-refseq.gtf
```

Then we specify the annotation file that we want it to use in order to identify how many reads are overlapping with known genomic features. We provide it with a file containing gene annotations, and we can save that file path to a variable, such as <code>gtf_file</code>, which will point to <code>~/_shared/bcourses-1557888-readonly/rna-seq/rna-feature/hg19-refseq.gtf</code>. So we can instead call up the variable as:

```bash
"${gtf_file}"
```

And including the option:

```bash
-a "${gtf_file}"
```

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
-o "${output_parent}/counts/1M_g03_counts.txt"
```

We can then specify the name of the output file along with the output path.


<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

```bash
${bam_files[@]}
```

After we specify our options, we then provide the alignment files that we want to obtain read counts for. Here, we can provide it with our BAM files that were sorted by name by expanding our variable <code>bam_files</code>. This will allow us to supply our BAM files for our control and our TAZ KO samples.

<hr style="border: 1px solid; border-color: #AAAAAA;"></hr>

<code>featureCounts</code> will then generate two output files. The first output file will contain the count table for your contrl and TAZ KO samples, while the other will be a file ending in the extension <code>.summary</code>. This second file will contain summarized count information for your samples, which we can then pull in for QC.

<a href="https://subread.sourceforge.net/SubreadUsersGuide.pdf" rel="noopener noreferrer"><u>Detailed information on the output tables can be found in the <code>featureCounts</code> documentation.</u></a>

The count table contains annotation columns and data columns that correspond to each sample that you provided the <code>featureCounts</code> program. 

<table style="text-align: center;">
    <tr style="font-size: 15px;">
        <th>Geneid</th>
        <th>Chr</th>
        <th>Start</th>
        <th>End</th>
        <th>Strand</th>
        <th>Length</th>
        <th>your control sample counts</th>
        <th>your tazko sample counts</th>
    </tr>
</table>

And the most relevant columns for our purposes are the <code>Geneid</code> column and the columns containing the counts for our samples.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise Set 1</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

For this exercise set, you'll explore the GTF annotation file to see what kind of information is contained within it, and then you'll also perform a quick QC to determine if there were any issues with your read counting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

<h2>Q1: Import <code>sample-gtf-to-explore.txt</code></h2>

Here, load in the GTF rows that we pulled out from our original GTF file to explore a bit. Recall that the table is tab-delimited.

During import, you can add the following headers in the below order:
<ol>
    <li>seqname</li>
    <li>source</li>
    <li>feature</li>
    <li>start</li>
    <li>end</li>
    <li>score</li>
    <li>strand</li>
    <li>frame</li>
    <li>attribute</li>
</ol>

Pull out the first 5 rows to take a look at the information contained within the GTF file.

What are the columns that are relevant for our purposes to use this file for counting reads?

Why might a single gene have multiple rows, and what does this tell us about how the data is structured?

<h2>Q2: Load in your group's counts summary table</h2>

Import the summary file that you have for your control and TAZ KO samples. The output from <code>featureCounts</code> is tab-delimited with column headers.

The values within the sample columns indicate how many reads that were assigned to those categories (<code>Status</code>). <a href="https://subread.sourceforge.net/SubreadUsersGuide.pdf" rel="noopener noreferrer"><u>Detailed information on what each <code>Status</code> is can be found in the documentation for <code>featureCounts</code></u></a> on page 36 and 37.

<h2>Q3: Update labels to remove <code>-name.bam</code></h2>

To tidy things up a bit, go ahead and remove the <code>-name.bam</code> from the column headers for each of your samples.

<h2>Q4: Generate a stacked bar plot using your read summary information</h2>

Now generate a stacked bar plot that visualizes the proportion/percent of each category stacked on top of each other. You can set up your plot so that each sample condition is its own stacked bar.

This will allow us to then do a quick QC to see what what the proportion of each category is to get an idea of whether or not something may have gone wrong during counting.

Recall back in our data visualization notebooks how to set up a stacked bar plot using <code>plt.bar()</code>. 

First set the <code>Status</code> column as the index for your DataFrame.

Then subset your DataFrame to only obtain the categories that have non-zero values.

Transpose the DataFrame to change the orientation to make it easier to use when creating a stacked bar plot.

Calculate all the values as a percentage for their respective sample.

Once that's all set up, go ahead and plot a stacked bar plot.

We'll reconvene here and take a look at how everyone's counts data looks like overall.

<h1 style="font-size: 40px; margin-bottom: 0px;">Exercise Set 2</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

For this exercise set, you'll take a look at your group's data to see how the two sample conditions compare to one another to assess whether or not there is potentially a detectable difference between them. 

And at the same time while you are performing QC, you will also create a tidied up <code>csv</code> file for us to use later on when we perform differential expression analysis. 

<h2>Q1: Import your read counts table and remove <code>-name.bam</code> from your sample column headers</h2>

Go ahead and import your counts <code>.txt</code> file. You'll want to keep in mind that the file has a comment line marked with the <code>#</code>, and it is also tab-delimited. Like with your summary table, go ahead and also remove the <code>-name.bam</code> part of the column headers for your sample columns.

<h2>Q2: Remove unneeded columns and export as a new <code>csv</code> file</h2>

We really only need the <code>Geneid</code> and sample columns for differential expression analysis, so go ahead and remove the extra annotation columns and export the tidied counts matrix as <code>clean_{SAMPLE}_counts.csv</code>.

Jack will then pull your cleaned counts file from your server to put into the shared directory for everyone to use later.

<h2>Q3: Plot sample conditions against each other</h2>

Now, generate a scatter plot where each dot corresponds to a single gene, and its position along the x-axis is determined by its read count in your control sample and its position along the y-axis is determined by its read count in your TAZ KO sample. If the data is difficult to distinguish due to the presence of particular outliers, perform a log transformation on your read counts or plot on log axes.

Consider what it means when a dot (gene) falls along the x=y axis vs showing some particular offset from x=y. What does its position in your scatter plot tell you about its expression in control vs TAZ KO cells?

<h2>Q4: Visualize potentially downregulated and upregulated genes</h2>

With your single replicate, you can take a quick look to see if there may be differential expression between your sample groups. For this exercise, use the same scatter plot set up you used in Q3, but highlight those genes who have a 2-fold difference in their read counts in your TAZ KO sample compared to your control sample.

Something to note is that these exercise sets are just from a single replicate based off of non-normalized counts, so we can't draw hard conclusions from this visualization, but it can help us get a sense of how our counts went.

Next week, we'll pull together our class counts and run a differential expression analysis to explore our results.

<h1 style="font-size: 40px; margin-bottom: 0px;">References</h1>

<hr style="margin-left: 0px; border: 0.25px solid; border-color: #000000; width: 98%;"></hr>

<a href="https://academic.oup.com/bioinformatics/article-lookup/doi/10.1093/bioinformatics/btt656" rel="noopener noreferrer"><u>Liao et al 2014 Bioinformatics:</u></a> featureCounts: an efficient general purpose program for assigning sequence reads to genomic features

<a href="https://academic.oup.com/bioinformatics/article/31/2/166/2366196" rel="noopener noreferrer"><u>Anders et al 2015 Bioinformatics:</u></a> HTSeq—a Python framework to work with high-throughput sequencing data

<a href="https://academic.oup.com/bioinformatics/article/38/10/2943/6551247" rel="noopener noreferrer"><u>Putri et al 2022 Bioinformatics:</u></a> Analysing high-throughput sequencing data in Python with HTSeq 2.0